<a href="https://colab.research.google.com/github/guilhermelaviola/DataScienceApplicationsInComputerVision/blob/main/Class16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Future Trends & Innovations in Computer Vision & Data Science**
Modern computer vision and data science are being revolutionized by self-supervised learning models like SimCLR and BYOL, which bypass the need for costly manual labeling by learning directly from vast amounts of raw data. This progress is fueled by hardware innovations like TPUs and NPUs, alongside the adaptation of Transformer architectures to visual tasks, enabling real-time processing on devices ranging from servers to smartphones. However, these advancements bring significant ethical challenges; for instance, historical audits have revealed that facial recognition systems can exhibit error rates up to 34.4% for darker-skinned females compared to just 0.8% for lighter-skinned males, highlighting the urgent need for diverse datasets and rigorous auditing. To ensure these technologies remain sustainable and inclusive, the field is moving toward interdisciplinary collaboration, energy-efficient techniques like model quantization, and privacy-focused frameworks such as federated learning.

In [1]:
# Importing all the necessary libraries and resources:
import torch
import torchvision.transforms as transforms
from PIL import Image

## **Example: Data Transformation Pipeline with PyTorch**
Following is a simplified conceptual example using PyTorch to demonstrate how you might set up a data transformation pipeline for a model like SimCLR. The model learns that these different views of the same image are "the same" object, even without a label telling it what the object is. The model (ResNet or Transformer) processes both views. The goal is to maximize the similarity between view_1 and view_2. While minimizing similarity with other images in the batch.

In [2]:
# Defining Augmentations (The core of Self-Supervised Learning):
simclr_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [3]:
# 'Contrastive' Dataset wrapper
class ContrastiveDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __getitem__(self, index):
        path = self.image_paths[index]
        img = Image.open(path).convert('RGB')

        # Creating two different augmented versions of the SAME image:
        view_1 = simclr_transform(img)
        view_2 = simclr_transform(img)

        return view_1, view_2

    def __len__(self):
        return len(self.image_paths)

In [4]:
# Conceptual Training Loop Snippet:
def train_step(model, view_1, view_2, optimizer):
    optimizer.zero_grad()

    # Getting vector representations (embeddings):
    out_1 = model(view_1)
    out_2 = model(view_2)

    # Contrastive Loss (e.g., NT-Xent) would go here. It rewards the model for making out_1 and out_2 very close in vector space:
    loss = contrastive_loss_function(out_1, out_2)

    loss.backward()
    optimizer.step()